<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/forest_area_class_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Forest Threshold & Type Analysis: Case Study
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [1]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────


In [2]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ connected to drive


In [ ]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

In [5]:
import ee

# ── CELL 4: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')

treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded
✅ treecover2000 masked


In [ ]:
# ── CELL 5: import libraries and data from data_config ─────────────────────
import shutil
import geemap
import pandas as pd
import time
from data_config import (FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup,
                         forest_bins, country_thresholds, state_thresholds, forestClasses
                        )


from gee_functions import (
    # Section 1: Forest area by threshold — countries
    prepare_forest_collection,
    export_forest_area,
    # Section 2: Forest area by threshold — states
    prepare_states_forest_collection,
    export_states_forest_area,
    # Section 3: Forest cover bins — countries
    get_forest_cover_bins_one_country,
    export_forest_cover_bins_all_countries,
    # Section 4: Forest cover bins — states
    get_forest_cover_bins_one_state,
    export_forest_cover_bins_all_states,
    # Section 5: GLC forest type area — countries
    get_forest_cover_area_type_country,
    export_forest_cover_area_type_all_countries,
    # Section 6: GLC forest type area — states
    get_forest_cover_area_type_state,
    export_forest_cover_area_type_all_states,
    #section 7 : area for each bin for each GLC forest type
    get_forest_area_bin_type_state,
    export_forest_area_bin_type_all_states,
    copy_gee_exports_to_repo,
  )

import gee_functions
gee_functions.treecover2000_masked = treecover2000_masked
gee_functions.datamask             = datamask
gee_functions.glc_2000             = glc_2000


print('✅ GEE objects injected into gee_functions')
print('✅ All GEE functions imported')
print('✅ Libraries imported')
print('✅ Data config loaded')

In [ ]:
# ── CELL 6: Run Exports — United States All States ────────────────────────────

#print('── Submitting forest area by threshold task ──')
#state_threshold_task = export_states_forest_area(us_state_names, state_thresholds)

#print('\n── Submitting GLC forest type area task ──')
#state_glc_task = export_forest_cover_area_type_all_states(us_state_names, forestClasses)
#print('\n── Submitting forest cover bins task ──')
#state_bins_task = export_forest_cover_bins_all_states(us_state_names, forest_bins)

# Oregon only
#export_forest_area_bin_type_all_states(['Oregon'], forest_bins, forestClasses, region_label='Oregon')

# All states
#export_forest_area_bin_type_all_states(us_state_names, forest_bins, forestClasses, region_label='all_states')

# Pacific Northwest states
#export_forest_area_bin_type_all_states(['Oregon', 'Washington', 'California'], forest_bins, forestClasses, region_label='pacific_northwest')

In [7]:
# ── CELL 7: Load, clean and save each file ───────────────────────────────────
GEE_FOLDER  = '/content/drive/MyDrive/GEE_exports/'
DATA_FOLDER = '/content/Biochar_forest_estimation/data/'
os.makedirs(DATA_FOLDER, exist_ok=True)

# --- Load, clean, rename columns, and save states_forest_area_all_thresholds.csv ---
states_forest_area_all_thresholds = pd.read_csv(GEE_FOLDER + 'states_forest_area_all_thresholds.csv')
states_forest_area_all_thresholds = states_forest_area_all_thresholds.sort_values(by=['NAME', 'threshold'])
states_forest_area_all_thresholds.rename(columns={'NAME': 'state', 'sum': 'area_Mha'}, inplace=True)
states_forest_area_all_thresholds.to_csv(DATA_FOLDER + 'states_forest_area_all_thresholds.csv', index=False)
print(f'✅ states_forest_area_all_thresholds: {len(states_forest_area_all_thresholds)} rows')
print(states_forest_area_all_thresholds.columns)

# --- Load, clean, rename columns, and save forest_cover_bins_all_states.csv ---
forest_cover_bins_all_states = pd.read_csv(GEE_FOLDER + 'forest_cover_bins_all_states.csv')
forest_cover_bins_all_states = forest_cover_bins_all_states.sort_values(by=['NAME', 'bin'])
forest_cover_bins_all_states.rename(columns={'NAME': 'state', 'sum': 'area_Mha'}, inplace=True)
forest_cover_bins_all_states.to_csv(DATA_FOLDER + 'forest_cover_bins_all_states.csv', index=False)
print(f'✅ forest_cover_bins_all_states: {len(forest_cover_bins_all_states)} rows')

# --- Load, clean, rename columns, and save forest_cover_area_type_all_states.csv ---
forest_cover_area_type_all_states = pd.read_csv(GEE_FOLDER + 'forest_cover_area_type_all_states.csv')
forest_cover_area_type_all_states = forest_cover_area_type_all_states.sort_values(by=['NAME'])
forest_cover_area_type_all_states.rename(columns={'NAME': 'state'}, inplace=True)
forest_cover_area_type_all_states.to_csv(DATA_FOLDER + 'forest_cover_area_type_all_states.csv', index=False)
print(f'✅ forest_cover_area_type_all_states: {len(forest_cover_area_type_all_states)} rows')


# --- Pivot the states_forest_area_all_thresholds DataFrame to a wide format --

states_forest_area_all_thresholds_wide = states_forest_area_all_thresholds.pivot(
    index='state',
    columns='threshold',
    values='area_Mha'
)

states_forest_area_all_thresholds_wide.columns = [f'threshold_{c}' for c in states_forest_area_all_thresholds_wide.columns]
states_forest_area_all_thresholds_wide = states_forest_area_all_thresholds_wide.reset_index()
print(states_forest_area_all_thresholds_wide.head())

states_forest_area_all_thresholds_wide.to_csv(DATA_FOLDER + 'states_forest_area_all_thresholds_wide.csv', index=False)

# --- Pivot the forest_cover_bins_all_states_wide DataFrame to a wide format ---# --- Rename columns of the wide DataFrame for clarity ---

forest_cover_bins_all_states_wide = forest_cover_bins_all_states.pivot(
    index='state',
    columns='bin',
    values='area_Mha'
)

forest_cover_bins_all_states_wide = forest_cover_bins_all_states_wide.reset_index()
print(forest_cover_bins_all_states_wide.head())
forest_cover_bins_all_states_wide.to_csv(DATA_FOLDER + 'forest_cover_bins_all_states_wide.csv', index=False)

✅ states_forest_area_all_thresholds: 250 rows
Index(['state', 'threshold', 'area_Mha'], dtype='object')
✅ forest_cover_bins_all_states: 9 rows
✅ forest_cover_area_type_all_states: 1 rows
        state  threshold_10  threshold_20  threshold_30  threshold_40  \
0     Alabama      9.616855      9.385398      9.125929      8.899155   
1      Alaska     56.799190     52.745037     50.205783     46.130349   
2     Arizona      2.961880      2.416421      1.863394      1.515884   
3    Arkansas      8.149206      7.931670      7.718344      7.521119   
4  California     12.996085     11.124684      9.903539      9.037235   

   threshold_50  
0      8.682462  
1     37.039620  
2      1.192494  
3      7.337972  
4      8.072230  
bin   state         10-20          20-30          30-40         40-50  \
0    Oregon  891614.91445  673441.312165  467278.591569  663502.21808   

bin         50-60          60-70          70-80         80-90        90-100  
0    455218.08482  333483.445558  695478.

In [ ]:
#print the files list in DATA folder
files = os.listdir(DATA_FOLDER)
print(files)

In [ ]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)